# ✅ Notebook 4 — Validate Full Pothole AI Pipeline

**Loads models from Drive:**
- `yolov8x-seg-pothole.pt`
- `midas_dpt_large_pothole_finetuned.pt`
- `siamese_resnet18_repair.pt`

## What this notebook does
1. Load all three trained models.
2. Upload sample road images.
3. Run end-to-end pipeline:
   - pothole detection (YOLO-seg)
   - depth estimation (MiDaS)
   - severity scoring (rule-based)
   - repair verification (Siamese)
4. Export deployment bundle zip.

> Run in order. If model files are missing in Drive, the notebook tells you exactly which one is missing.

In [ ]:
# ── 1) Check GPU ─────────────────────────────────────────────────────────────
!nvidia-smi
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
# ── 2) Install dependencies ──────────────────────────────────────────────────
!pip install -q --upgrade ultralytics opencv-python-headless albumentations
print('✅ Dependencies installed')

In [ ]:
# ── 3) Mount Drive + paths ───────────────────────────────────────────────────
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

MODEL_DIR = Path('/content/drive/MyDrive/pothole_models')
YOLO_PATH = MODEL_DIR / 'yolov8x-seg-pothole.pt'
MIDAS_PATH = MODEL_DIR / 'midas_dpt_large_pothole_finetuned.pt'
SIAMESE_PATH = MODEL_DIR / 'siamese_resnet18_repair.pt'

print('Model dir:', MODEL_DIR)

In [ ]:
# ── 4) Verify required files exist ───────────────────────────────────────────
required = [YOLO_PATH, MIDAS_PATH, SIAMESE_PATH]
missing = [p for p in required if not p.exists()]

if missing:
    print('❌ Missing model files:')
    for p in missing:
        print(' -', p)
    raise FileNotFoundError('Upload/copy missing files to Drive before continuing.')

print('✅ All model files found')
for p in required:
    print(' -', p.name)

In [ ]:
# ── 5) Load YOLO detector ────────────────────────────────────────────────────
from ultralytics import YOLO

yolo_model = YOLO(str(YOLO_PATH))
print('✅ YOLO loaded')

In [ ]:
# ── 6) Load MiDaS depth model ────────────────────────────────────────────────
import torch

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
midas = torch.hub.load('intel-isl/MiDaS', 'DPT_Large').to(device)
state = torch.load(MIDAS_PATH, map_location=device)
midas.load_state_dict(state, strict=False)
midas.eval()

print('✅ MiDaS loaded on', device)

In [ ]:
# ── 7) Load Siamese verifier ─────────────────────────────────────────────────
import torch.nn as nn
import torchvision.models as models

class SiameseNetwork(nn.Module):
    def __init__(self):
        super().__init__()
        backbone = models.resnet18(weights=None)
        self.encoder = nn.Sequential(*list(backbone.children())[:-1])
        self.head = nn.Sequential(
            nn.Linear(512, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(0.2),
            nn.Linear(256, 1),
            nn.Sigmoid(),
        )

    def embed(self, x):
        return self.encoder(x).flatten(1)

    def forward(self, x1, x2):
        z1 = self.embed(x1)
        z2 = self.embed(x2)
        d = torch.abs(z1 - z2)
        return self.head(d)

siamese = SiameseNetwork().to(device)
siamese.load_state_dict(torch.load(SIAMESE_PATH, map_location=device), strict=False)
siamese.eval()
print('✅ Siamese loaded')

In [ ]:
# ── 8) Pipeline helper functions ─────────────────────────────────────────────
import cv2
import numpy as np


def estimate_depth_cm(img_bgr):
    rgb = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    x = cv2.resize(rgb, (384, 384), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
    x = (x - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
    x = torch.from_numpy(np.transpose(x, (2, 0, 1))).unsqueeze(0).float().to(device)
    with torch.no_grad():
        d = midas(x)
        if d.ndim == 3:
            d = d.unsqueeze(1)
    depth_map = d[0, 0].detach().cpu().numpy()
    depth_map = (depth_map - depth_map.min()) / (depth_map.max() - depth_map.min() + 1e-6)
    est_depth_cm = float(depth_map.mean() * 20.0)
    return depth_map, est_depth_cm


def classify_severity(area_ratio, depth_cm):
    score = (0.55 * area_ratio) + (0.45 * min(depth_cm / 15.0, 1.0))
    if score < 0.33:
        return 'LOW', score
    if score < 0.66:
        return 'MEDIUM', score
    return 'HIGH', score


def siamese_score(img_a_bgr, img_b_bgr):
    def prep(im):
        im = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        im = cv2.resize(im, (224, 224), interpolation=cv2.INTER_LINEAR).astype(np.float32) / 255.0
        im = (im - np.array([0.485, 0.456, 0.406], np.float32)) / np.array([0.229, 0.224, 0.225], np.float32)
        return torch.from_numpy(np.transpose(im, (2, 0, 1))).unsqueeze(0).float().to(device)

    x1, x2 = prep(img_a_bgr), prep(img_b_bgr)
    with torch.no_grad():
        s = float(siamese(x1, x2).item())
    if s >= 0.85:
        label = 'repaired'
    elif s >= 0.60:
        label = 'partially_repaired'
    else:
        label = 'not_repaired'
    return s, label

In [ ]:
# ── 9) Upload test images ────────────────────────────────────────────────────
from google.colab import files as colab_files
from pathlib import Path

TEST_DIR = Path('/content/test_images')
TEST_DIR.mkdir(parents=True, exist_ok=True)

print('Upload 1+ test road images (.jpg/.png):')
uploaded = colab_files.upload()

test_images = []
for name, data in uploaded.items():
    p = TEST_DIR / name
    p.write_bytes(data)
    if p.suffix.lower() in ['.jpg', '.jpeg', '.png']:
        test_images.append(p)

print(f'✅ Loaded {len(test_images)} test image(s)')
if len(test_images) == 0:
    raise RuntimeError('No test images uploaded.')

In [ ]:
# ── 10) Run full pipeline on uploaded images ─────────────────────────────────
import matplotlib.pyplot as plt

pipeline_results = []

for img_path in test_images:
    img_bgr = cv2.imread(str(img_path))
    if img_bgr is None:
        continue

    # A) Detect potholes
    det = yolo_model.predict(source=img_bgr, imgsz=1024, conf=0.55, iou=0.45, verbose=False)
    r = det[0]
    num_det = 0 if r.boxes is None else len(r.boxes)

    area_ratio = 0.0
    if r.masks is not None and len(r.masks.data) > 0:
        mask = r.masks.data[0].detach().cpu().numpy()
        area_ratio = float(mask.sum() / mask.size)
    elif r.boxes is not None and len(r.boxes) > 0:
        x1, y1, x2, y2 = r.boxes.xyxy[0].detach().cpu().numpy()
        area_ratio = float(max(0, (x2 - x1) * (y2 - y1)) / (img_bgr.shape[0] * img_bgr.shape[1]))

    # B) Depth
    depth_map, depth_cm = estimate_depth_cm(img_bgr)

    # C) Severity
    severity, sev_score = classify_severity(area_ratio, depth_cm)

    # D) Repair verification (self-vs-smoothed proxy)
    repaired_proxy = cv2.GaussianBlur(img_bgr, (9, 9), 0)
    sim_score, repair_status = siamese_score(img_bgr, repaired_proxy)

    pipeline_results.append({
        'image': img_path.name,
        'detections': int(num_det),
        'area_ratio': area_ratio,
        'depth_cm': depth_cm,
        'severity': severity,
        'severity_score': float(sev_score),
        'repair_similarity': sim_score,
        'repair_status': repair_status,
    })

    # Visual
    vis = r.plot() if hasattr(r, 'plot') else cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(12, 4))
    plt.subplot(1, 3, 1); plt.imshow(cv2.cvtColor(img_bgr, cv2.COLOR_BGR2RGB)); plt.title(img_path.name); plt.axis('off')
    plt.subplot(1, 3, 2); plt.imshow(vis); plt.title(f'Detections: {num_det}'); plt.axis('off')
    plt.subplot(1, 3, 3); plt.imshow(depth_map, cmap='magma'); plt.title(f'Depth~{depth_cm:.1f} cm'); plt.axis('off')
    plt.tight_layout(); plt.show()

print('✅ Pipeline run complete')
for row in pipeline_results:
    print(row)

In [ ]:
# ── 11) Export deployment bundle ─────────────────────────────────────────────
import json, shutil
from pathlib import Path
from google.colab import files

BUNDLE_DIR = Path('/content/pothole_models_bundle')
if BUNDLE_DIR.exists():
    shutil.rmtree(BUNDLE_DIR)
BUNDLE_DIR.mkdir(parents=True, exist_ok=True)

shutil.copy(YOLO_PATH, BUNDLE_DIR / 'yolov8x-seg-pothole.pt')
shutil.copy(MIDAS_PATH, BUNDLE_DIR / 'midas_dpt_large_pothole_finetuned.pt')
shutil.copy(SIAMESE_PATH, BUNDLE_DIR / 'siamese_resnet18_repair.pt')

(BUNDLE_DIR / 'pipeline_validation_results.json').write_text(json.dumps(pipeline_results, indent=2))

zip_base = '/content/pothole_models_bundle'
zip_file = shutil.make_archive(zip_base, 'zip', root_dir=BUNDLE_DIR)

print('✅ Bundle ready:', zip_file)
print('Includes 3 model files + validation results JSON')
files.download(zip_file)